In [5]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [6]:
def load_corpus(data_dir):
    texts = []
    # Đếm tổng số file trước để tqdm có thể hiển thị tiến trình
    all_files = []
    for root, dirs, files in os.walk(data_dir):
        for fname in files:
            if fname.endswith('.txt'):
                all_files.append(os.path.join(root, fname))
    
    print("Đang load corpus...")
    for fpath in tqdm(all_files, desc="Loading files"):
        with open(fpath, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if text:
                texts.append(text)
    return texts

train_dir = r"d:\dut_ai\AIO_code\Dence Representation\data\data_train\train"
corpus = load_corpus(train_dir)
print(f"Số documents: {len(corpus)}")

Đang load corpus...


Loading files: 100%|██████████| 30000/30000 [02:06<00:00, 237.22it/s] 

Số documents: 30000


In [7]:
def preprocess(texts, min_freq=3):
    all_tokens = []
    tokenized_docs = []
    print("Đang tiền xử lý dữ liệu...")
    for text in tqdm(texts, desc="Preprocessing"):
        text = text.lower()
        tokens = re.findall(r'[a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ_]+', text)
        tokenized_docs.append(tokens)
        all_tokens.extend(tokens)
    
    freq = Counter(all_tokens)
    vocab = ['<UNK>'] + [w for w, c in freq.items() if c >= min_freq]
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    
    tokenized_docs = [
        [w if w in word2idx else '<UNK>' for w in doc]
        for doc in tokenized_docs
    ]
    return tokenized_docs, vocab, word2idx, idx2word

tokenized_docs, vocab, word2idx, idx2word = preprocess(corpus, min_freq=3)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

Đang tiền xử lý dữ liệu...


Preprocessing: 100%|██████████| 30000/30000 [00:02<00:00, 10486.38it/s]


Vocab size: 13676


In [8]:
def generate_skipgram_data(tokenized_docs, word2idx, window=2):
    data = []
    print("Đang sinh dữ liệu Skip-gram...")
    for tokens in tqdm(tokenized_docs, desc="Generating pairs"):
        ids = [word2idx[w] for w in tokens]
        for i in range(len(ids)):
            center_id = ids[i]
            start = max(0, i - window)
            end = min(len(ids), i + window + 1)
            for j in range(start, end):
                if i == j: continue
                context_id = ids[j]
                data.append((center_id, context_id))
    return data

data = generate_skipgram_data(tokenized_docs, word2idx, window=2)
print(f"Số cặp Skip-gram: {len(data)}")

Đang sinh dữ liệu Skip-gram...


Generating pairs: 100%|██████████| 30000/30000 [00:06<00:00, 4599.32it/s]

Số cặp Skip-gram: 9684018


In [9]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(SkipGram, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear    = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, center_id):
        embed = self.embedding(center_id) 
        out   = self.linear(embed)       
        return out

In [10]:
EMBED_DIM = 100
EPOCHS = 5
LR = 0.001
BATCH_SIZE = 1024
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SkipGram(vocab_size, EMBED_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Subset data và split Train/Val
SUBSET_SIZE = min(len(data), 1000000)
data_subset = data[:SUBSET_SIZE]
train_data, val_data = train_test_split(data_subset, test_size=0.2, random_state=42)

def create_loader(data_list, batch_size):
    X = torch.tensor([c for c, ctx in data_list], dtype=torch.long)
    y = torch.tensor([ctx for c, ctx in data_list], dtype=torch.long)
    ds = torch.utils.data.TensorDataset(X, y)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)

train_loader = create_loader(train_data, BATCH_SIZE)
val_loader = create_loader(val_data, BATCH_SIZE)

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for x_batch, y_batch in pbar_train:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        batch_loss = loss.item()
        train_loss += batch_loss * x_batch.size(0)
        _, predicted = torch.max(output, 1)
        train_correct += (predicted == y_batch).sum().item()
        train_total += y_batch.size(0)
        
        pbar_train.set_postfix({"loss": f"{batch_loss:.4f}"})
        
    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
    with torch.no_grad():
        for x_batch, y_batch in pbar_val:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output = model(x_batch)
            loss = criterion(output, y_batch)
            
            val_loss += loss.item() * x_batch.size(0)
            _, predicted = torch.max(output, 1)
            val_correct += (predicted == y_batch).sum().item()
            val_total += y_batch.size(0)
            
    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Summary Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}\n")

Epoch 1/5 [Val]: 100%|██████████| 196/196 [00:18<00:00, 10.61it/s]


Summary Epoch 1: Train Loss: 7.6099, Train Acc: 0.0237 | Val Loss: 6.7217, Val Acc: 0.0346



Epoch 2/5 [Val]: 100%|██████████| 196/196 [00:18<00:00, 10.69it/s]


Summary Epoch 2: Train Loss: 6.5036, Train Acc: 0.0374 | Val Loss: 6.5248, Val Acc: 0.0391



Epoch 3/5 [Val]: 100%|██████████| 196/196 [00:19<00:00,  9.80it/s]


Summary Epoch 3: Train Loss: 6.3165, Train Acc: 0.0408 | Val Loss: 6.4593, Val Acc: 0.0412



Epoch 4/5 [Val]: 100%|██████████| 196/196 [00:18<00:00, 10.41it/s]


Summary Epoch 4: Train Loss: 6.2111, Train Acc: 0.0429 | Val Loss: 6.4294, Val Acc: 0.0427



Epoch 5/5 [Val]: 100%|██████████| 196/196 [00:17<00:00, 11.21it/s]

Summary Epoch 5: Train Loss: 6.1378, Train Acc: 0.0444 | Val Loss: 6.4145, Val Acc: 0.0436



In [11]:
def get_similarity(word, model, word2idx, idx2word, top_k=5):
    if word not in word2idx: return "Từ không có trong vocab"
    idx = torch.tensor([word2idx[word]]).to(device)
    embed = model.embedding(idx).detach()
    all_embeds = model.embedding.weight.detach()
    norm_word = embed / embed.norm(dim=1, keepdim=True)
    norm_all = all_embeds / all_embeds.norm(dim=1, keepdim=True)
    sim = torch.mm(norm_word, norm_all.T).squeeze()
    values, indices = torch.topk(sim, top_k + 1)
    results = []
    for i in range(1, len(indices)):
        results.append((idx2word[indices[i].item()], values[i].item()))
    return results

print("Các từ tương tự với 'ngon':")
print(get_similarity('ngon', model, word2idx, idx2word))

Các từ tương tự với 'ngon':
[('bất_kể', 0.402038037776947), ('qá', 0.3669252097606659), ('đèn', 0.36358344554901123), ('chình', 0.36255475878715515), ('phồng_tôm', 0.36116477847099304)]
